# Experiment: CyEmbed Archetype Embedding Sweep (Breast CyTOF + VIM)


Notebook-first training and hyperparameter sweep for CyTOF archetype models.

This notebook supports two decoders:
- `factorized`: `W -> H = WZ -> X_hat = H E^T + b`
- `direct`: `X_hat = W A`

Edit the config cell and run top-to-bottom.


In [8]:
from __future__ import annotations

from pathlib import Path
from itertools import islice
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CyEmbed.data import extract_matrix, fit_scaler, preprocess_array, split_train_val_indices
from CyEmbed.train import build_sweep_configs, run_sweep
from CyEmbed.utils import collect_software_versions, save_json, set_seed

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [9]:
# === Editable configuration ===
OUTPUT_ROOT = Path("outputs/breast_vim_archetype_embedding_sweep")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(
    f"Using OUTPUT_ROOT: {OUTPUT_ROOT.resolve()} "
    "(existing runs are preserved; completed runs are skipped)"
)

GLOBAL_CFG = {
    "seed": 7,
    "deterministic": True,
    "device": "auto",  # auto resolves CUDA -> MPS -> CPU
}

DATA_CFG = {
    # Input mode: "adata", "dataframe", or "matrix"
    "input_mode": "adata",

    # Optional path-based loading (set one according to input_mode)
    "adata_path": "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/data/breast_cytof_withvim_processed.h5ad",
    "dataframe_path": None,
    "matrix_npy_path": None,

    # AnnData extraction options
    "adata_source": "X",     # "X", "layer", "obsm"
    "adata_layer": None,
    "adata_obsm_key": None,
    "sample_col": "sample_id",
    "cluster_col": None,

    # Dataset-specific cleanup/filtering
    "drop_rank_obs_columns": True,
    "filter_nan_label": False,

    # Fallback for dry-runs when no data path/object is supplied
    "allow_synthetic_if_missing": False,
    "synthetic_n_cells": 8000,
    "synthetic_n_markers": 35,
    "synthetic_n_samples": 8,
    "synthetic_n_clusters": 6,
}

PREPROCESS_CFG = {
    "mode": "zscore",  # "none", "zscore", "robust_zscore"
    "balanced_scaling_by_sample": False,
    "balanced_max_per_sample": 2000,
}

SPLIT_CFG = {
    "val_fraction": 0.2,
    "stratify_by": "sample",  # None, "sample", or "cluster"
    "train_size": 50000,
}

BASE_TRAIN_CFG = {
    "epochs": 800,
    "early_stopping": True,
    "patience": 20,
    "min_delta": 0.0,
    "restore_best_weights": True,
    "weight_decay": 1e-5,
    "dropout": 0.0,
    "huber_delta": 1.0,
    "separation_mode": "cosine_sq",  # "cosine_mean", "cosine_abs", "cosine_sq", "rbf"
    "balance_mode": "l2_uniform",    # "l2_uniform", "kl_uniform", "neg_entropy"
    "rbf_gamma": 1.0,
    "print_every": 10,
    "progress_sweep": True,
    "skip_existing_runs": True,
    "progress_epoch": True,
}

SWEEP_GRID = {
    # Requested sweep focus: lower K range + embedding dimension sweep.
    "decoder_type": ["factorized"],
    "K": [4, 5, 6, 7, 8],
    "d": [4, 6, 8, 12],
    "hidden_dims": [[256, 128]],
    "lr": [1e-3],
    "batch_size": [2048],  # try 1024/512 if you hit OOM
    "recon_loss_type": ["mse"],
    "lambda_entropy": [1e-3],
    "lambda_sep": [1e-3],
    "lambda_balance": [5e-2],
    "tau": [1.0],
}

MAX_RUNS = None


In [10]:
# === Data loading ===
# You can provide data in any of these ways:
# 1) Set DATA_CFG paths above.
# 2) Predefine one of `adata_obj`, `prepared_df`, `prepared_matrix` in the notebook.

set_seed(GLOBAL_CFG["seed"], deterministic=GLOBAL_CFG["deterministic"])

adata_obj = globals().get("adata_obj", None)
prepared_df = globals().get("prepared_df", None)
prepared_matrix = globals().get("prepared_matrix", None)

bundle = None

if DATA_CFG["input_mode"] == "adata":
    adata = adata_obj
    if adata is None and DATA_CFG["adata_path"] is not None:
        import anndata as ad
        adata = ad.read_h5ad(DATA_CFG["adata_path"])

    if adata is not None:
        obs_cols = pd.Index(adata.obs.columns)
        label_like_cols = [
            c for c in obs_cols
            if any(tok in str(c).lower() for tok in ["label", "cluster", "celltype", "cell_type"])
        ]
        print(f"Detected label-like obs columns: {label_like_cols}")

        if DATA_CFG.get("drop_rank_obs_columns", False):
            rank_cols = [c for c in obs_cols if str(c).lower().endswith("_rank")]
            if rank_cols:
                adata.obs = adata.obs.drop(columns=rank_cols)
                print(f"Dropped rank obs columns: n={len(rank_cols)}")

        resolved_cluster_col = DATA_CFG.get("cluster_col")
        if resolved_cluster_col is None and len(label_like_cols) > 0:
            resolved_cluster_col = label_like_cols[0]
        print(f"Using cluster column: {resolved_cluster_col}")

        bundle = extract_matrix(
            adata=adata,
            source=DATA_CFG["adata_source"],
            layer=DATA_CFG["adata_layer"],
            obsm_key=DATA_CFG["adata_obsm_key"],
            sample_col=DATA_CFG["sample_col"],
            cluster_col=resolved_cluster_col,
        )

        if DATA_CFG.get("filter_nan_label", False):
            if bundle.cluster_ids is None:
                print("filter_nan_label=True but no cluster labels are available; skipping filter.")
            else:
                label_series = pd.Series(bundle.cluster_ids)
                keep_mask = (~label_series.isna()).to_numpy()
                removed = int((~keep_mask).sum())
                if removed > 0:
                    bundle.X = bundle.X[keep_mask]
                    bundle.cell_ids = [cid for cid, keep in zip(bundle.cell_ids, keep_mask) if keep]
                    if bundle.sample_ids is not None:
                        bundle.sample_ids = bundle.sample_ids[keep_mask]
                    if bundle.cluster_ids is not None:
                        bundle.cluster_ids = bundle.cluster_ids[keep_mask]
                print(f"Filtered NaN labels: removed={removed}, kept={bundle.X.shape[0]}")

elif DATA_CFG["input_mode"] == "dataframe":
    df = prepared_df
    if df is None and DATA_CFG["dataframe_path"] is not None:
        df = pd.read_csv(DATA_CFG["dataframe_path"], index_col=0)
    if df is not None:
        bundle = extract_matrix(dataframe=df)

elif DATA_CFG["input_mode"] == "matrix":
    matrix = prepared_matrix
    if matrix is None and DATA_CFG["matrix_npy_path"] is not None:
        matrix = np.load(DATA_CFG["matrix_npy_path"])
    if matrix is not None:
        bundle = extract_matrix(matrix=matrix)

else:
    raise ValueError("DATA_CFG['input_mode'] must be one of: adata/dataframe/matrix")

if bundle is None:
    if not DATA_CFG["allow_synthetic_if_missing"]:
        raise ValueError("No input data found. Provide AnnData/DataFrame/matrix in config.")

    print("No input data supplied. Generating synthetic CyTOF-like matrix for dry-run.")
    n_cells = DATA_CFG["synthetic_n_cells"]
    n_markers = DATA_CFG["synthetic_n_markers"]
    n_samples = DATA_CFG["synthetic_n_samples"]
    n_clusters = DATA_CFG["synthetic_n_clusters"]

    rng = np.random.default_rng(GLOBAL_CFG["seed"])
    z = rng.normal(size=(n_cells, 6)).astype(np.float32)
    loadings = rng.normal(size=(6, n_markers)).astype(np.float32)
    x = z @ loadings + 0.2 * rng.normal(size=(n_cells, n_markers)).astype(np.float32)

    sample_ids = np.array([f"sample_{i}" for i in rng.integers(0, n_samples, size=n_cells)])
    cluster_ids = np.array([f"cluster_{i}" for i in rng.integers(0, n_clusters, size=n_cells)])

    bundle = extract_matrix(
        matrix=x,
        marker_names=[f"marker_{i}" for i in range(n_markers)],
        cell_ids=[f"cell_{i}" for i in range(n_cells)],
        sample_ids=sample_ids,
        cluster_ids=cluster_ids,
    )

print(f"Loaded X shape: {bundle.X.shape}")
print(f"Sample metadata present: {bundle.sample_ids is not None}")
print(f"Cluster metadata present: {bundle.cluster_ids is not None}")


Detected label-like obs columns: []
Using cluster column: None
Loaded X shape: (150000, 31)
Sample metadata present: True
Cluster metadata present: False


In [11]:
# === Preprocess and split ===
fit_sample_ids = bundle.sample_ids if PREPROCESS_CFG["balanced_scaling_by_sample"] else None
balanced_max = PREPROCESS_CFG["balanced_max_per_sample"] if PREPROCESS_CFG["balanced_scaling_by_sample"] else None

scaler, scaler_fit_idx = fit_scaler(
    bundle.X,
    mode=PREPROCESS_CFG["mode"],
    sample_ids=fit_sample_ids,
    balanced_max_per_sample=balanced_max,
    random_state=GLOBAL_CFG["seed"],
)
X_proc = preprocess_array(bundle.X, scaler)

if SPLIT_CFG["stratify_by"] == "sample" and bundle.sample_ids is not None:
    stratify_labels = bundle.sample_ids
elif SPLIT_CFG["stratify_by"] == "cluster" and bundle.cluster_ids is not None:
    stratify_labels = bundle.cluster_ids
else:
    stratify_labels = None

train_idx, val_idx = split_train_val_indices(
    n_cells=X_proc.shape[0],
    val_fraction=SPLIT_CFG["val_fraction"],
    seed=GLOBAL_CFG["seed"],
    stratify_labels=stratify_labels,
)

target_train = SPLIT_CFG.get("train_size")
if target_train is not None:
    target_train = int(target_train)
    if len(train_idx) > target_train:
        rng = np.random.default_rng(GLOBAL_CFG["seed"])
        train_idx = np.sort(rng.choice(train_idx, size=target_train, replace=False))
    elif len(train_idx) < target_train:
        print(f"Requested train_size={target_train}, but only {len(train_idx)} train cells are available.")

print(f"Scaler mode: {scaler.mode}")
print(f"Scaler fit cells: {len(scaler_fit_idx)}")
print(f"Train cells: {len(train_idx)} | Val cells: {len(val_idx)}")


Scaler mode: zscore
Scaler fit cells: 150000
Train cells: 50000 | Val cells: 30000


In [ ]:
# === Build sweep and train ===
sweep_configs = build_sweep_configs(SWEEP_GRID)
if MAX_RUNS is not None:
    sweep_configs = sweep_configs[: int(MAX_RUNS)]

print(f"Total sweep runs: {len(sweep_configs)}")

notebook_config = {
    "global": GLOBAL_CFG,
    "data": DATA_CFG,
    "preprocess": PREPROCESS_CFG,
    "split": SPLIT_CFG,
    "base_train": BASE_TRAIN_CFG,
    "sweep_grid": SWEEP_GRID,
    "software_versions": collect_software_versions(),
    "scaler": scaler.to_dict(),
}
save_json(OUTPUT_ROOT / "notebook_config.json", notebook_config)

summary_df = run_sweep(
    x=X_proc,
    marker_names=bundle.marker_names,
    cell_ids=bundle.cell_ids,
    output_root=OUTPUT_ROOT,
    base_config={**GLOBAL_CFG, **BASE_TRAIN_CFG},
    sweep_configs=sweep_configs,
    train_idx=train_idx,
    val_idx=val_idx,
    sample_ids=bundle.sample_ids,
    cluster_ids=bundle.cluster_ids,
    scaler_state=scaler.to_dict(),
)

summary_df.to_csv(OUTPUT_ROOT / "sweep_summary_sorted.csv", index=False)
summary_df.head(20)


Total sweep runs: 20


Sweep:   0%|          | 0/20 [00:00<?, ?it/s]

[run001_20260416_100526] Using device: mps


run001_20260416_100526 epochs:   0%|          | 0/800 [00:00<?, ?it/s]

[run001_20260416_100526] Early stopping at epoch 451 (patience=20, best_epoch=431, best_val_recon=0.52008)
[run002_20260416_100813] Using device: mps


...

run018_20260416_104935 epochs:   0%|          | 0/800 [00:00<?, ?it/s]

[run018_20260416_104935] Early stopping at epoch 413 (patience=20, best_epoch=393, best_val_recon=0.36640)
[run019_20260416_105209] Using device: mps


run019_20260416_105209 epochs:   0%|          | 0/800 [00:00<?, ?it/s]

[run019_20260416_105209] Early stopping at epoch 497 (patience=20, best_epoch=477, best_val_recon=0.33577)
[run020_20260416_105528] Using device: mps


run020_20260416_105528 epochs:   0%|          | 0/800 [00:00<?, ?it/s]

In [ ]:
# === Sweep summary plots ===
if 'summary_df' not in globals():
    summary_df = pd.read_csv(OUTPUT_ROOT / "sweep_summary.csv")

display_cols = [
    "run_id", "decoder_type", "K", "d", "hidden_dims", "lr", "batch_size",
    "recon_loss_type", "lambda_entropy", "lambda_sep", "lambda_balance", "tau",
    "val_recon", "mean_marker_corr_val", "mean_entropy_val",
    "dominant_frac_gt_0_5_val", "dominant_frac_gt_0_8_val", "run_dir"
]
summary_view = summary_df[display_cols].sort_values("val_recon", ascending=True)
display(summary_view)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(summary_df["val_recon"], summary_df["mean_marker_corr_val"], alpha=0.8)
axes[0].set_xlabel("Val reconstruction loss")
axes[0].set_ylabel("Mean marker Pearson r")
axes[0].set_title("Reconstruction vs correlation")

axes[1].scatter(summary_df["tau"], summary_df["mean_entropy_val"], alpha=0.8)
axes[1].set_xlabel("Tau")
axes[1].set_ylabel("Mean entropy(W)")
axes[1].set_title("Softmax temperature effect")

for decoder, group in summary_df.groupby("decoder_type"):
    axes[2].scatter(group["val_recon"], group["dominant_frac_gt_0_8_val"], alpha=0.8, label=decoder)
axes[2].set_xlabel("Val reconstruction loss")
axes[2].set_ylabel("Fraction dominant > 0.8")
axes[2].set_title("Purity by decoder")
axes[2].legend(frameon=False)

plt.tight_layout()

best_row = summary_df.iloc[0]
print("Best run:")
display(best_row.to_frame().T)

best_history = pd.read_csv(Path(best_row["run_dir"]) / "history.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(best_history["epoch"], best_history["loss"], label="train_total")
ax.plot(best_history["epoch"], best_history["val_recon"], label="val_recon")
ax.set_xlabel("Epoch")
ax.set_title(f"History: {best_row['run_id']}")
ax.legend(frameon=False)
plt.tight_layout()
